# 09 임베딩 기반 시계열 분석

PCA, DTW, AutoEncoder, GAF-CNN, TS2Vec, PatchTST 6가지 임베딩 후 GMM 클러스터링 품질(실루엣, DB Index)을 비교합니다.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED, ML_CLUSTER
from utils.metrics import clustering_quality
from utils.embeddings import EMBEDDERS

dfw = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
pivot = dfw.pivot_table(index=['type','family'], columns='yearweek', values='sales', fill_value=0)
meta = pivot.index.to_frame(index=False)
X_raw = StandardScaler().fit_transform(pivot.values)
print('panel:', X_raw.shape)

In [ ]:
K = 4
results = []
best = None
best_score = -np.inf

for name, fn in EMBEDDERS.items():
    print('embedding:', name)
    X_emb = fn(X_raw, n_components=10)
    gmm = GaussianMixture(n_components=K, random_state=42, n_init=10)
    labels = gmm.fit_predict(X_emb)
    q = clustering_quality(X_emb, labels)
    q['embedding'] = name
    q['clustering'] = 'GMM'
    results.append(q)

    score = (q['silhouette'] if not np.isnan(q['silhouette']) else -1) - (q['davies_bouldin'] if not np.isnan(q['davies_bouldin']) else 0)
    if score > best_score:
        best_score = score
        best = (name, X_emb, labels)

quality_df = pd.DataFrame(results)[['embedding','clustering','n_clusters','silhouette','davies_bouldin']]
print('=== 임베딩×GMM 품질 ===')
print(quality_df.round(4))

In [ ]:
best_name, X_best, labels_best = best
print('최적 임베딩:', best_name)

out = meta.copy()
out['ML_CLUSTER'] = labels_best + 1
out['embedding_method'] = best_name
out['clustering_method'] = 'GMM'
out.to_parquet(ML_CLUSTER, index=False)

quality_df.to_csv(DATA_PROCESSED / 'embedding_clustering_quality.csv', index=False)
np.save(DATA_PROCESSED / 'embedding_matrix.npy', X_best)
print('저장:', ML_CLUSTER)
out.head()